# 11.2 Lineare Regression

In Kapitel 11.1 haben wir gelernt, Daten aus DataFrames direkt mit Plotly
Express zu visualisieren. Streudiagramme zeigen dabei oft einen Trend: Mit
steigender Motorleistung steigt der Preis, mit steigendem Kilometerstand fällt
er. Die **lineare Regression** macht diesen Trend mathematisch greifbar. Sie
sucht die Gerade, die den beobachteten Zusammenhang am besten beschreibt, und
liefert mit dem Bestimmtheitsmaß R² eine Zahl dafür, wie gut die Gerade zu den
Daten passt.

## Lernziele

* [ ] Sie können mit `np.polyfit()` eine Regressionsgerade berechnen und die
  Koeffizienten interpretieren.
* [ ] Sie können mit `np.poly1d()` eine aufrufbare Geradenfunktion erzeugen.
* [ ] Sie können das **Bestimmtheitsmaß R²** mit `np.corrcoef()` berechnen und
  interpretieren.
* [ ] Sie können eine Regressionsgerade mit `fig.add_scatter()` in ein
  Streudiagramm einzeichnen.

## Geradenfit mit `np.polyfit()`

Eine Gerade hat die Form $y = m \cdot x + b$, wobei $m$ die Steigung und $b$
der y-Achsenabschnitt ist. `np.polyfit(x, y, 1)` berechnet die Werte von $m$
und $b$, sodass die Gerade möglichst nah an allen Datenpunkten liegt. Das Argument
`1` steht für den Grad des Polynoms; für eine Gerade ist das immer `1`.

`np.polyfit()` erwartet numerische eindimensionale Daten *ohne* fehlende Werte.
Aus einem DataFrame extrahieren wir die Spalten daher mit `.values` und
entfernen vorher alle Zeilen mit fehlenden Werten in den relevanten Spalten mit
`.dropna(subset=[...])`.

Wir laden den bereinigten Datensatz und bereiten die Daten für die Regression vor:

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px

# Eingabe
df = pd.read_csv("autoscout24_DE_2020_cleaned.csv")

df_benzin = df[(df["Kraftstoff"] == "Benzin") &
               (df["Preis (Euro)"] <= 100000)].dropna(subset=["Leistung (kW)",
                                                               "Preis (Euro)"])

x = df_benzin["Leistung (kW)"].values
y = df_benzin["Preis (Euro)"].values

Mit `np.polyfit()` berechnen wir die Koeffizienten der Regressionsgeraden:

In [ ]:
# Verarbeitung
koeff = np.polyfit(x, y, 1)
print(koeff)

Das Ergebnis ist ein Array mit zwei Werten: `koeff[0]` ist die Steigung $m$,
`koeff[1]` ist der y-Achsenabschnitt $b$. Wir geben die Koeffizienten
interpretierbar aus:

In [ ]:
m = koeff[0]
b = koeff[1]
print(f"Steigung:           {m:.1f} Euro pro kW")
print(f"y-Achsenabschnitt:  {b:.0f} Euro")

Die Steigung von etwa 171 Euro pro kW bedeutet: Jeder zusätzliche Kilowatt
Motorleistung ist im Mittel mit etwa 171 Euro mehr Kaufpreis verbunden. Das ist
eine konkrete, interpretierbare Aussage.

Mit `np.poly1d()` erzeugen wir aus den Koeffizienten eine aufrufbare Funktion,
ähnlich wie wir in Kapitel 8 eigene Funktionen definiert haben:

In [ ]:
gerade = np.poly1d(koeff)

print(f"Preis bei  80 kW:  {gerade(80):.0f} Euro")
print(f"Preis bei 150 kW:  {gerade(150):.0f} Euro")
print(f"Preis bei 200 kW:  {gerade(200):.0f} Euro")

`gerade(x)` liefert den vorhergesagten Preis für eine gegebene Leistung. Da
`gerade` intern mit NumPy arbeitet, kann man ihr auch ein Array übergeben und
erhält für jeden Eingabewert einen Vorhersagewert zurück.

### Mini-Übung 1

Führen Sie dieselbe Regression für **Dieselfahrzeuge** durch (Preis bis
100.000 Euro). Geben Sie Steigung und y-Achsenabschnitt aus und vergleichen Sie
die Steigung mit dem Ergebnis für Benzinfahrzeuge. Was bedeutet der Unterschied?

In [ ]:
# Code-Zelle

## Bestimmtheitsmaß R²

Die Regressionsgerade wird immer berechnet, egal wie gut oder schlecht sie zu
den Daten passt. Das **Bestimmtheitsmaß R²** misst, wie gut die Gerade den
Zusammenhang tatsächlich erklärt. Es liegt zwischen 0 und 1:

* R² = 1: Die Gerade erklärt die Daten vollständig, alle Punkte liegen exakt auf
  der Geraden.
* R² = 0: Die Gerade erklärt nichts, es gibt keinen linearen Zusammenhang.
* R² = 0.75: Die Gerade erklärt etwa 75 % der Varianz der Preise im Rahmen eines
  linearen Modells. Der Rest der Streuung geht auf andere Faktoren oder
  Nichtlinearitäten zurück.

Wir berechnen R² über den Korrelationskoeffizienten $r$, den `np.corrcoef()`
liefert. Der Rückgabewert ist eine 2×2-Matrix; der Korrelationskoeffizient
steht an Position `[0, 1]`:

In [ ]:
# Eingabe und Verarbeitung wie oben
df = pd.read_csv("autoscout24_DE_2020_cleaned.csv")
df_benzin = df[(df["Kraftstoff"] == "Benzin") &
               (df["Preis (Euro)"] <= 100000)].dropna(subset=["Leistung (kW)",
                                                               "Preis (Euro)"])
x = df_benzin["Leistung (kW)"].values
y = df_benzin["Preis (Euro)"].values

# Verarbeitung
r = np.corrcoef(x, y)[0, 1]
R2 = r**2

# Ausgabe
print(f"r  = {r:.3f}")
print(f"R² = {R2:.3f}")

R² ≈ 0.75 bedeutet: Die Motorleistung erklärt 75 % der Preisstreuung. Das ist
ein starker linearer Zusammenhang, aber 25 % der Variation bleiben durch andere
Faktoren wie Marke, Ausstattung oder Kilometerstand unbeschrieben.

### Hohe Korrelation ist keine Kausalität

Als weiteres Beispiel berechnen wir R² für den Zusammenhang zwischen
Kraftstoffverbrauch in l/100 km und CO₂-Ausstoß in g/km. Wir filtern auf
physikalisch plausible Werte, da der Rohdatensatz vereinzelte Fehleingaben
enthält:

In [ ]:
# Eingabe
df_verbrauch = df[(df["Kraftstoff"] == "Benzin") &
                  (df["Verbrauch (l/100 km)"] >= 3) &
                  (df["Verbrauch (l/100 km)"] <= 20) &
                  (df["Verbrauch (g/km)"] >= 70) &
                  (df["Verbrauch (g/km)"] <= 470)].dropna(
                      subset=["Verbrauch (l/100 km)", "Verbrauch (g/km)"])

x_v = df_verbrauch["Verbrauch (l/100 km)"].values
y_v = df_verbrauch["Verbrauch (g/km)"].values

# Verarbeitung
r_v = np.corrcoef(x_v, y_v)[0, 1]
koeff_v = np.polyfit(x_v, y_v, 1)

# Ausgabe
print(f"R² = {r_v**2:.3f}")
print(f"Steigung: {koeff_v[0]:.2f} g/km pro l/100 km")

R² ≈ 0.96 ist eine sehr hohe Korrelation. Die Steigung von etwa 22.5 g/km pro
l/100 km entspricht dem physikalischen Umrechnungsfaktor für Benzin: Jeder
verbrannte Liter Kraftstoff erzeugt eine feste Menge CO₂.

Beide Größen messen dieselbe physikalische Ursache: die Verbrennung von
Kraftstoff. Das hohe R² zeigt die enge Verwandtschaft, sagt aber nichts darüber
aus, welche Größe die andere verursacht. Würde man durch eine Gesetzgebung den
CO₂-Wert in den Fahrzeugdaten auf null setzen, sänke der Kraftstoffverbrauch
dadurch nicht um einen Milliliter. Hohe Korrelation bedeutet immer nur, dass
zwei Größen zusammenhängen, nie dass eine die andere bewirkt.

### Mini-Übung 2

Berechnen Sie R² für den Zusammenhang zwischen `Leistung (kW)` und
`Leistung (PS)` im gesamten Datensatz (alle Kraftstoffarten, keine Preisfilter).
Was erwarten Sie? Interpretieren Sie das Ergebnis.

In [ ]:
# Code-Zelle

## Regressionsgerade im Streudiagramm

Bisher haben wir Regressionsgerade und Datenpunkte getrennt betrachtet. Jetzt
zeichnen wir beides in ein gemeinsames Diagramm. Der Scatter-Plot entsteht wie
in Kapitel 11.1 mit `px.scatter()`. Mit der Methode `.add_scatter()` fügen wir
der Abbildung eine weitere Kurve hinzu:

In [ ]:
# Eingabe
df_benzin = df[(df["Kraftstoff"] == "Benzin") &
               (df["Preis (Euro)"] <= 100000)].dropna(subset=["Leistung (kW)",
                                                               "Preis (Euro)"])

x = df_benzin["Leistung (kW)"].values
y = df_benzin["Preis (Euro)"].values

# Verarbeitung: Regression
koeff = np.polyfit(x, y, 1)
gerade = np.poly1d(koeff)
R2 = np.corrcoef(x, y)[0, 1]**2

x_fit = np.linspace(x.min(), x.max(), 200)
y_fit = gerade(x_fit)

# Ausgabe: Scatter mit Regressionsgerade
fig = px.scatter(data_frame=df_benzin,
                 x="Leistung (kW)",
                 y="Preis (Euro)",
                 opacity=0.3,
                 title=f"Preis vs. Leistung (Benzin) | R² = {R2:.2f}")

fig.add_scatter(x=x_fit,
                y=y_fit,
                mode="lines",
                name=f"Regression (m = {koeff[0]:.0f} €/kW)")
fig.show()

`x_fit` ist ein gleichmäßig verteiltes Array von 200 Punkten zwischen dem
kleinsten und größten x-Wert, erzeugt mit `np.linspace()` aus Kapitel 9.1.
`gerade(x_fit)` berechnet für jeden dieser Punkte den vorhergesagten Preis.
`opacity=0.3` macht die Datenpunkte halbtransparent, damit die Regressionsgerade
besser sichtbar ist. R² erscheint direkt im Titel, damit die Güte der Anpassung
sofort ablesbar ist.

### Mini-Übung 3

Zeichnen Sie die Regressionsgerade für den Zusammenhang zwischen
`Verbrauch (l/100 km)` und `Verbrauch (g/km)` für Benzinfahrzeuge. Verwenden
Sie die gefilterten Daten aus dem R²-Abschnitt (3 bis 20 l/100 km, 70 bis
470 g/km). Geben Sie R² und die Steigung im Titel aus.

In [ ]:
# Code-Zelle

## Zusammenfassung und Ausblick

In diesem Kapitel haben wir die lineare Regression als Werkzeug kennengelernt,
um Zusammenhänge in Daten zu quantifizieren. `np.polyfit()` berechnet Steigung
und y-Achsenabschnitt, `np.poly1d()` macht die Gerade aufrufbar, und
`np.corrcoef()` liefert mit R² ein Maß für die Güte der Anpassung. Die
Visualisierung mit `fig.add_scatter()` verbindet die Regression mit den
Diagrammtypen aus Kapitel 11.1. Das Verbrauch-Beispiel hat gezeigt, dass ein
hohes R² zwar einen engen Zusammenhang anzeigt, aber keine Aussage über
Ursache und Wirkung erlaubt. Pandas, Plotly Express und NumPy bilden zusammen
ein leistungsfähiges Werkzeugset für die Datenanalyse, das in weiterführenden
Veranstaltungen zu maschinellem Lernen und technischer Datenanalyse ausgebaut
werden kann.